# 🌐 Web Intelligence Pipeline — Sovereign Edge Vector Augmentation
**Legion-Jacked-Pipeline | Web Intel Layer**

This notebook mirrors the exact 3-Lane Delta Architecture from `headless_audio_processing-github.ipynb`, 
but targets **live web API data** instead of raw `.wav` files.

The core thesis: by augmenting our 37-dimensional acoustic Omni-Vectors with structured web metadata 
(BPM, genre, label, ISRC, play counts), we dramatically increase the reasoning density of every LanceDB vector.

> **Lane 1 → DuckDB (WebDB):** Raw API JSON payloads, immutably stored as physical truth  
> **Lane 2 → LanceDB (VectorDB):** Snowflake Arctic embeddings of the combined acoustic + web payload  
> **Lane 3 → H.O.R.N. Daemon:** Structured audit logs and system event tracking


## ⚙️ CELL 1 — Imports & Sovereign Edge Configuration

In [ ]:
import httpx
import json
import duckdb
import lancedb
import ollama
import sys
import os
from datetime import datetime
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, Field
import pydantic

# ── Force UTF-8 for Windows terminals ──────────────────────
sys.stdout.reconfigure(encoding='utf-8')

# ── Sovereign Edge Configuration ───────────────────────────
OLLAMA_HOST        = "http://127.0.0.1:11434"
OLLAMA_EMBED_MODEL = "snowflake-arctic-embed:latest"   # Same embed model, same 1024-dim space

DB_PATH          = "./web_intel_sonicdb.duckdb"        # Lane 1: Physical truth
LANCEDB_PATH     = "./lancedb_web_intel_rag"           # Lane 2: Vector space
TABLE_NAME       = "web_intel_tracks"

# ── Target API ─────────────────────────────────────────────
# Swap this URL out for any API target (Apple Music, Discogs, SoundCloud, etc.)
GITHUB_REPO = "adamscarmccoy-boop/ADAMSCARMCCOY-CASE_STUDIES"

print(f"Sovereign Edge Web Intel Pipeline initialized.")
print(f"Embed Model : {OLLAMA_EMBED_MODEL}")
print(f"DuckDB Path : {DB_PATH}")
print(f"LanceDB Path: {LANCEDB_PATH}")


## 🏗️ CELL 2 — Pydantic Schema Definitions (3-Lane Delta Architecture)

In [ ]:
# ── LANE 1: DuckDB Raw Web Payload Schema ──────────────────
class Lane1WebDBPayload(BaseModel):
    """Lane 1: Immutable raw web API payload — physical truth, never modified"""
    source_url:    str
    endpoint_type: str   # e.g. 'github_api', 'apple_music', 'discogs'
    track_title:   Optional[str] = None
    artist_name:   Optional[str] = None
    bpm:           Optional[float] = None
    genre:         Optional[str] = None
    label:         Optional[str] = None
    isrc:          Optional[str] = None
    release_date:  Optional[str] = None
    play_count:    Optional[int] = None
    raw_json:      str           # Full API response archived verbatim
    ingested_at:   str = Field(default_factory=lambda: datetime.now().isoformat())

# ── LANE 2: LanceDB Omni-Vector (Acoustic + Web Augmented) ─
class Lane2AugmentedOmniVector(BaseModel):
    """
    Lane 2: Snowflake Arctic embedding of the full multi-modal payload.
    This is the money shot: acoustic physics + web metadata = max reasoning density.
    """
    segment_name:     str

    # ── Acoustic Dimensions (from headless_audio_processing-github.ipynb) ──
    spectral_centroid:  float   # Normalized: raw_hz / 10000.0
    rms:                float   # Normalized: raw * 10.0
    zero_crossing_rate: float   # Normalized: raw * 10.0
    spectral_bandwidth: float   # Normalized: raw_hz / 10000.0
    mfcc_vector:        List[float]

    # ── Web Metadata Augmentation Dimensions ───────────────────────────────
    bpm:           Optional[float] = None
    genre:         Optional[str]  = None
    label:         Optional[str]  = None
    isrc:          Optional[str]  = None
    release_year:  Optional[int]  = None

    # ── Snowflake Vector (1024-dim) ────────────────────────────────────────
    vector: Optional[List[float]] = None

    @pydantic.model_validator(mode='after')
    def enforce_vector_multipliers(self) -> 'Lane2AugmentedOmniVector':
        """Enforces raw-to-scaled multiplier transitions (same as audio pipeline)"""
        if self.spectral_centroid > 10.0:
            self.spectral_centroid = self.spectral_centroid / 10000.0
        if self.spectral_bandwidth > 10.0:
            self.spectral_bandwidth = self.spectral_bandwidth / 10000.0
        if self.rms < 1.0:
            self.rms = self.rms * 10.0
        if self.zero_crossing_rate < 1.0:
            self.zero_crossing_rate = self.zero_crossing_rate * 10.0
        return self

# ── LANE 3: H.O.R.N. System Log ────────────────────────────
class Lane3HornLog(BaseModel):
    """Lane 3: Structured audit log for every pipeline event"""
    process_id:        int
    execution_severity: str  # INFO | WARNING | CRITICAL
    root_cause_analysis: str
    timestamp:         str = Field(default_factory=lambda: datetime.now().isoformat())

print("3-Lane Delta Pydantic schemas loaded.")


## 🗄️ CELL 3 — Lane 1: DuckDB WebDB Initialization (SonicDB Equivalent)

In [ ]:
# Initialize the WebDB — the immutable raw API truth store
conn = duckdb.connect(DB_PATH)

conn.execute("""
    CREATE TABLE IF NOT EXISTS web_intel_raw (
        id              INTEGER PRIMARY KEY,
        source_url      TEXT,
        endpoint_type   TEXT,
        track_title     TEXT,
        artist_name     TEXT,
        bpm             FLOAT,
        genre           TEXT,
        label           TEXT,
        isrc            TEXT,
        release_date    TEXT,
        play_count      INTEGER,
        raw_json        TEXT,
        ingested_at     TEXT
    )
""")

print(f"WebDB initialized at: {DB_PATH}")
row_count = conn.execute("SELECT COUNT(*) FROM web_intel_raw").fetchone()[0]
print(f"Current rows in WebDB: {row_count}")


## 🌐 CELL 4 — XHR/API Interception Layer (GitHub API Target)

In [ ]:
# ── XHR Interception: Hit the raw API, not the HTML page ──
# This is the same philosophy as reading raw Librosa bytes instead of rendered waveforms.
# We bypass all HTML parsing and go straight to the structured JSON.

HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

def fetch_repo_metadata(repo: str) -> dict:
    """Fetches raw GitHub repo metadata — swap this function for any API target"""
    url = f"https://api.github.com/repos/{repo}"
    r = httpx.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

def fetch_repo_commits(repo: str, limit: int = 10) -> list:
    """Fetches latest commit history — rich metadata for vector augmentation"""
    url = f"https://api.github.com/repos/{repo}/commits"
    r = httpx.get(url, headers=HEADERS, params={"per_page": limit}, timeout=10)
    r.raise_for_status()
    return r.json()

# ── Pull the raw data ───────────────────────────────────────
print(f"Intercepting API for: {GITHUB_REPO}")
repo_meta   = fetch_repo_metadata(GITHUB_REPO)
commits     = fetch_repo_commits(GITHUB_REPO)

print(f"Repo: {repo_meta.get('full_name')}")
print(f"Stars: {repo_meta.get('stargazers_count')}  |  Forks: {repo_meta.get('forks_count')}")
print(f"Language: {repo_meta.get('language')}")
print(f"Latest commit: {commits[0]['commit']['message'][:80] if commits else 'N/A'}")
print(f"Total commits fetched: {len(commits)}")


## 💾 CELL 5 — Lane 1 Flush: Archive Raw JSON to DuckDB WebDB

In [ ]:
# Archive the raw API payload verbatim — this is our physical truth store.
# Just like SonicDB stores the raw Librosa acoustics, WebDB stores the raw API JSON.

payloads_to_insert = []

for commit in commits:
    payload = Lane1WebDBPayload(
        source_url    = commit.get("url", ""),
        endpoint_type = "github_api",
        track_title   = commit["commit"]["message"][:120],
        artist_name   = commit["commit"]["author"]["name"],
        release_date  = commit["commit"]["author"]["date"],
        raw_json      = json.dumps(commit)
    )
    payloads_to_insert.append(payload)

# Bulk insert into DuckDB
for i, p in enumerate(payloads_to_insert):
    conn.execute("""
        INSERT INTO web_intel_raw 
        (id, source_url, endpoint_type, track_title, artist_name, release_date, raw_json, ingested_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, [i, p.source_url, p.endpoint_type, p.track_title, p.artist_name, p.release_date, p.raw_json, p.ingested_at])

conn.commit()
print(f"Flushed {len(payloads_to_insert)} raw API payloads to WebDB (Lane 1)")
conn.execute("SELECT id, track_title, artist_name FROM web_intel_raw LIMIT 5").df()


## ❄️ CELL 6 — Lane 2: Snowflake Arctic Embedding (Augmented Omni-Vectors)

In [ ]:
# ── The Money Shot: Build the Augmented Multi-Modal Vector ─
# We serialize the full combined payload (web metadata + acoustic stub) into a text string,
# then pass the WHOLE thing to Snowflake Arctic for a 1024-dim embedding.
# This is how we increase reasoning density: more data in → richer vector out.

client = ollama.Client(host=OLLAMA_HOST)

def build_embed_text(payload: Lane1WebDBPayload) -> str:
    """
    Serialize the full multi-modal payload into a text string for embedding.
    Snowflake Arctic will encode ALL of this context into 1024 dimensions.
    """
    return (
        f"commit_message: {payload.track_title} "
        f"author: {payload.artist_name} "
        f"date: {payload.release_date} "
        f"source: {payload.endpoint_type} "
        f"repo: {GITHUB_REPO} "
    )

def embed_payload(text: str) -> List[float]:
    """Get 1024-dim Snowflake Arctic embedding via local Ollama"""
    response = client.embeddings(model=OLLAMA_EMBED_MODEL, prompt=text)
    return response["embedding"]

print(f"Routing payloads to Snowflake Arctic on {OLLAMA_HOST}...")

augmented_vectors = []
for p in payloads_to_insert:
    embed_text = build_embed_text(p)
    vector     = embed_payload(embed_text)
    
    augmented_vectors.append({
        "segment_name": p.track_title[:60],
        "artist":       p.artist_name,
        "release_date": p.release_date,
        "source":       p.endpoint_type,
        "embed_text":   embed_text,
        "vector":       vector
    })

print(f"Generated {len(augmented_vectors)} augmented vectors")
print(f"Vector dimensionality: {len(augmented_vectors[0]['vector'])}")


## 🗃️ CELL 7 — Lane 2 Flush: Ingest Augmented Vectors into LanceDB

In [ ]:
import pyarrow as pa

# Connect to LanceDB and create/overwrite the web intel table
db = lancedb.connect(LANCEDB_PATH)

schema = pa.schema([
    pa.field("segment_name",  pa.string()),
    pa.field("artist",        pa.string()),
    pa.field("release_date",  pa.string()),
    pa.field("source",        pa.string()),
    pa.field("embed_text",    pa.string()),
    pa.field("vector",        pa.list_(pa.float32(), 1024)),
])

# Write to LanceDB
if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)
    
table = db.create_table(TABLE_NAME, data=augmented_vectors, schema=schema)

print(f"Web Intel vectors ingested into LanceDB: {LANCEDB_PATH}")
print(f"Table: '{TABLE_NAME}'  |  Rows: {table.count_rows()}")


## 🔍 CELL 8 — Semantic RAG Query: Multi-Modal Vector Search

In [ ]:
# ── Query the augmented vector space ────────────────────────
# The richer our vectors, the more precise the semantic search results.

def query_web_intel(query_text: str, top_k: int = 3):
    """
    Embed the query with Snowflake Arctic and retrieve the most semantically 
    similar web intel payloads from LanceDB.
    """
    query_vector = embed_payload(query_text)
    
    tbl     = db.open_table(TABLE_NAME)
    results = tbl.search(query_vector).limit(top_k).to_list()
    
    print(f"Query: '{query_text}'")
    print(f"Top {top_k} Results:")
    print("=" * 60)
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r['segment_name']}")
        print(f"       Author: {r['artist']}  |  Date: {r['release_date']}")
        print(f"       Source: {r['source']}")
        print()

# Run a query
query_web_intel("cognitive audio pipeline architecture fix")


## 📊 CELL 9 — DuckDB Analytics: Lane 1 Audit Report

In [ ]:
# ── Lane 1 Audit: Query the raw WebDB for analytics ────────
# Same pattern as the audio pipeline — raw data is always queryable in DuckDB.

print("WebDB Lane 1 Audit Report")
print("=" * 60)

df = conn.execute("""
    SELECT 
        endpoint_type,
        COUNT(*) as total_payloads,
        MIN(release_date) as earliest,
        MAX(release_date) as latest,
        COUNT(DISTINCT artist_name) as unique_authors
    FROM web_intel_raw
    GROUP BY endpoint_type
""").df()

print(df.to_string())
print()
print(f"Total raw payloads archived in WebDB: {conn.execute('SELECT COUNT(*) FROM web_intel_raw').fetchone()[0]}")
conn.close()
